# Data-Driven Insights for Sustainable Development Goals
### EPA112A Final Assignment 2026 - Group 17

| Name | Student number |
|---|---|
| Hazel Lim | 5697158 |
| Woobie Mizobuchi Reeh | 6593720 |

**Group number:** 17

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid")

DATA_DIR = Path("data")
START_YEAR, END_YEAR = 1960, 2025

## 1. Introduction

### Problem statement

Maternal and child mortality remain major public-health challenges, and their burdens differ considerably between countries. National wealth is an important part of this inequality: wealthier countries generally have more resources to support healthcare, sanitation, and other conditions associated with survival. Yet income alone may not capture how political institutions and social opportunities affect health outcomes.

Women's political empowerment offers another perspective. When women can exercise civil liberties, participate in civil society, and engage in political decision-making, their interests may be better represented in public life. However, whether countries with greater women's political empowerment also experience lower maternal and child mortality (and whether that relationship persists after accounting for national income) is an empirical question.

Our project is framed primarily by **SDG 3: Good Health and Well-being**, particularly **Target 3.1** (reducing maternal mortality) and **Target 3.2** (ending preventable deaths of newborns and children under five). It also connects to **SDG 5: Gender Equality**, especially women's participation in decision-making. The analysis focuses on political empowerment rather than treating it as a complete measure of gender equality.

### Main research question

**To what extent is women's political empowerment associated with maternal and under-five mortality across countries, after accounting for GDP per capita?**

### Subquestions

1. How is women's political empowerment associated with maternal mortality and under-five mortality across countries in 2022?
2. How do these relationships compare with the associations between GDP per capita and mortality?
3. Do the observed patterns remain similar in 2023, and does the association with empowerment persist when GDP per capita is included in a statistical model?

### Study design

We use 2022 as the primary cross-section and 2023 as a sensitivity check. The unit of observation is a country-year. The analysis is observational and descriptive/associational; it cannot establish that women's political empowerment causes changes in mortality. All countries with the necessary data are eligible, rather than restricting the sample to a particular income group.

## 2. Datasets

We combine two independent public data sources:

| Source | Indicator | Code / column | Unit | Role |
|---|---|---|---|---|
| World Bank, World Development Indicators (WDI) | Maternal mortality ratio | `SH.STA.MMRT` | Maternal deaths per 100,000 live births | Outcome 1 |
| World Bank, WDI | Under-five mortality rate | `SH.DYN.MORT` | Deaths before age five per 1,000 live births | Outcome 2 |
| World Bank, WDI | GDP per capita, PPP (constant international $) | `NY.GDP.PCAP.PP.KD` | International dollars per person | Income control |
| V-Dem, Country-Year Core v16 | Women's Political Empowerment Index | `v2x_gender` | Index from 0 to 1; higher = greater empowerment | Main explanatory variable |

**Data provenance:** [World Bank WDI](https://data.worldbank.org/) and [V-Dem dataset](https://www.v-dem.net/data/the-v-dem-dataset/). The V-Dem index measures women's **political empowerment** through civil liberties, civil society participation, and political participation. It should not be interpreted as a comprehensive index of gender equality.

The World Bank downloads contain countries **and** regional/income aggregates, which must be excluded. World Bank data arrive in **wide format** (one column per year), whereas V-Dem arrives in **long format** (one row per country-year). We merge by the three-letter country code and year, not by country names, because names can differ across sources.

**Temporal availability:** The source files contain many historical years. We retain 2000–2024 during preparation, but maternal mortality is unavailable for 2024 in the supplied release. Therefore, the complete-case analysis covers 2000–2023; 2022 and 2023 are used for the main comparison.

### Input files

- `API_SH.STA.MMRT_DS2_EN_csv_v2_2368.csv`
- `API_SH.DYN.MORT_DS2_EN_csv_v2_2145.csv`
- `API_NY.GDP.PCAP.PP.KD_DS2_EN_csv_v2_2481.csv`
- `V-Dem-CY-Core-v16.csv`

In [ ]:
%pip install pycountry

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import pycountry

In [ ]:
# Use the same folder structure for all group members.
DATA_DIR = Path("data/raw_data")
OUTPUT_DIR = Path("data/processed_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WB_FILES = {
    "maternal": "API_SH.STA.MMRT_DS2_EN_csv_v2_2368.csv",
    "under5": "API_SH.DYN.MORT_DS2_EN_csv_v2_2145.csv",
    "gdp_ppp": "API_NY.GDP.PCAP.PP.KD_DS2_EN_csv_v2_2481.csv",
}
VDEM_FILE = "V-Dem-CY-Core-v16.csv"
YEARS = list(range(2000, 2025))
MAIN_VARIABLES = ["v2x_gender", "maternal", "under5", "gdp_ppp"]

for filename in list(WB_FILES.values()) + [VDEM_FILE]:
    if not (DATA_DIR / filename).exists():
        raise FileNotFoundError(f"Missing original input: {DATA_DIR / filename}")

print("All four source files found.")

## 3. Preprocessing

The following steps are performed directly in Python:

1. **Exclude World Bank aggregates:** keep valid ISO three-letter country codes (plus `XKX`, used for Kosovo). We use `pycountry` for this identification rather than treating every World Bank row as a country.
2. **Reshape World Bank indicators:** convert year columns into a `year` column using `pandas.melt()`.
3. **Standardize merge keys:** rename the country-code fields to `country_code`, convert `year` to an integer, and convert measurements to numeric values.
4. **Select V-Dem fields:** retain the women's political empowerment index and its three component indices; restrict to the same year range.
5. **Merge:** perform one-to-one outer merges on `country_code` and `year` to preserve coverage information, including rows missing from a source.
6. **Audit:** check duplicate keys, missing values, and country coverage by year. We do **not** fill missing values with zero or impute missing mortality measurements.
7. **Create analysis samples:** retain only complete observations for all four main variables in 2022 and 2023. This selection can introduce coverage bias, which we will discuss later.

In [ ]:
# Country codes: prevent regional and income-group aggregates entering the analysis.
valid_country_codes = {country.alpha_3 for country in pycountry.countries}
valid_country_codes.add("XKX")  # Kosovo


def load_world_bank(filename, variable_name):
    """Read one original WDI CSV and return country-year observations."""
    wide = pd.read_csv(DATA_DIR / filename, skiprows=4, low_memory=False)
    wide = wide.loc[:, ~wide.columns.str.startswith("Unnamed:")].copy()
    wide = wide[wide["Country Code"].isin(valid_country_codes)].copy()

    year_columns = [str(year) for year in YEARS]
    if not set(year_columns).issubset(wide.columns):
        raise ValueError(f"Expected year columns missing from {filename}")

    long = wide.melt(
        id_vars=["Country Name", "Country Code"],
        value_vars=year_columns,
        var_name="year",
        value_name=variable_name,
    )
    long = long.rename(columns={
        "Country Name": "wb_country_name",
        "Country Code": "country_code",
    })
    long["year"] = long["year"].astype(int)
    long[variable_name] = pd.to_numeric(long[variable_name], errors="coerce")

    if long.duplicated(["country_code", "year"]).any():
        raise ValueError(f"Duplicate country-year keys in {filename}")
    return long


world_bank = {}
for variable, filename in WB_FILES.items():
    world_bank[variable] = load_world_bank(filename, variable)
    print(variable, world_bank[variable].shape,
          "nonmissing:", world_bank[variable][variable].notna().sum())

In [ ]:
# Standardize V-Dem keys and retain only the needed variables.
vdem_raw = pd.read_csv(DATA_DIR / VDEM_FILE, low_memory=False)

vdem = vdem_raw.rename(columns={"country_text_id": "country_code"}).copy()
vdem = vdem[
    vdem["country_code"].isin(valid_country_codes)
    & vdem["year"].between(min(YEARS), max(YEARS))
].copy()
vdem["year"] = vdem["year"].astype(int)

if vdem.duplicated(["country_code", "year"]).any():
    raise ValueError("Duplicate V-Dem country-year keys")

print("V-Dem countries:", vdem["country_code"].nunique())
print("V-Dem observations, 2000–2024:", len(vdem))

In [ ]:
# Outer merge keeps all available country-year rows for missingness auditing.
panel = vdem.copy()
for variable in ["maternal", "under5", "gdp_ppp"]:
    wb_values = world_bank[variable][["country_code", "year", variable]]
    panel = panel.merge(wb_values, on=["country_code", "year"],
                        how="outer", validate="one_to_one")

# Fill country names for countries appearing only in World Bank files.
wb_names = pd.concat([
    frame[["country_code", "wb_country_name"]]
    for frame in world_bank.values()
], ignore_index=True).drop_duplicates(subset="country_code")
panel = panel.merge(wb_names, on="country_code", how="left",
                    validate="many_to_one")
panel["country_name"] = panel["country_name"].fillna(panel["wb_country_name"])
panel = panel.drop(columns="wb_country_name")
panel = panel.sort_values(["country_code", "year"]).reset_index(drop=True)

panel["complete_case"] = panel[MAIN_VARIABLES].notna().all(axis=1)
panel["complete_core"] = panel[["v2x_gender", "maternal", "under5"]].notna().all(axis=1)

assert not panel.duplicated(["country_code", "year"]).any()
assert panel.loc[panel["complete_case"], MAIN_VARIABLES].notna().all().all()
print("Merged country-year rows:", len(panel))
print("Countries in merged panel:", panel["country_code"].nunique())
display(panel.head())

In [ ]:
# Audit coverage and missingness by year.
coverage_by_year = panel.groupby("year").agg(
    country_rows=("country_code", "size"),
    complete_cases=("complete_case", "sum"),
    **{f"{col}_available": (col, "count") for col in MAIN_VARIABLES},
).reset_index()

for col in MAIN_VARIABLES:
    coverage_by_year[f"{col}_missing_pct"] = (
        100 * (1 - coverage_by_year[f"{col}_available"] /
                 coverage_by_year["country_rows"])
    ).round(1)

display(coverage_by_year.tail(6))

# Missingness in the two primary cross-sectional samples before complete-case filtering.
for year in [2022, 2023]:
    subset = panel[panel["year"] == year]
    print(f"\n{year}: {len(subset)} country rows, "
          f"{subset['complete_case'].sum()} complete cases")
    display(subset[MAIN_VARIABLES].isna().sum().rename("missing_count").to_frame())

In [ ]:
# Country-level completeness across the full 2000–2024 preparation window.
coverage_by_country = panel.groupby(["country_code", "country_name"],
                                    dropna=False).agg(
    years_observed=("year", "size"),
    complete_years=("complete_case", "sum"),
    **{f"{col}_available": (col, "count") for col in MAIN_VARIABLES},
).reset_index()

# Create the primary and sensitivity-check samples.
data_2022 = panel[(panel["year"] == 2022) & panel["complete_case"]].copy()
data_2023 = panel[(panel["year"] == 2023) & panel["complete_case"]].copy()
complete_panel = panel[(panel["year"] <= 2023) & panel["complete_case"]].copy()

assert len(data_2022) == 167, "Unexpected 2022 count; check source versions"
assert len(data_2023) == 166, "Unexpected 2023 count; check source versions"
assert len(complete_panel) == 3974, "Unexpected panel count; check source versions"

print("2022 analysis sample:", len(data_2022), "countries")
print("2023 analysis sample:", len(data_2023), "countries")
print("Complete 2000–2023 observations:", len(complete_panel))

# Save outputs for subsequent EDA and regression cells.
panel.to_csv(OUTPUT_DIR / "women_clean_merged_panel_2000_2024.csv", index=False)
complete_panel.to_csv(OUTPUT_DIR / "women_complete_panel_2000_2023.csv", index=False)
data_2022.to_csv(OUTPUT_DIR / "women_2022.csv", index=False)
data_2023.to_csv(OUTPUT_DIR / "women_2023.csv", index=False)
coverage_by_year.to_csv(OUTPUT_DIR / "coverage_by_year.csv", index=False)
coverage_by_country.to_csv(OUTPUT_DIR / "coverage_by_country.csv", index=False)
print("Processed files saved to", OUTPUT_DIR)

### Preprocessing decisions and limitations

The cleaned dataset retains original numerical values; missing observations are not replaced with zeros or estimated values. The primary 2022 analysis includes **167 countries**, and the 2023 comparison includes **166 countries**, each with nonmissing values for all four main indicators. The full merged 2000–2024 file retains incomplete observations so that missingness remains visible. The complete-case 2000–2023 panel contains **3,974 country-year observations**.

Using an outer merge makes source coverage auditable but does not make all countries comparable: only complete-case rows enter the subsequent models. Country availability may be systematically related to income, governance, or health conditions, so the sample is not necessarily representative of every country. The 2022 and 2023 samples are not perfectly identical; comparisons between them may reflect changes in sample composition as well as indicator values.

We also retain extreme but valid-looking values at this stage. Potential outliers will be investigated in the EDA section rather than automatically removed. Later analyses will use logarithmic transformations for strictly positive mortality rates and GDP per capita to address skewness. No causal interpretation is made from these observational cross-country comparisons.

## 4. Data analysis

Raw mortality rates are strongly right-skewed (skewness 2.3 for maternal and 5.8 for under-5 mortality), whereas the log-transformed values are close to symmetric (-0.08 and 0.12). We therefore model log mortality and log GDP. Coefficients on log GDP are elasticities; coefficients on the empowerment index are log-point changes per one unit of the 0-1 index, which we report as percentage changes per 0.1 points.

In [ ]:
import statsmodels.formula.api as smf

# Log-transform skewed variables (all values are strictly positive)
for frame in (data_2022, data_2023, complete_panel):
    frame["log_maternal"] = np.log(frame["maternal"])
    frame["log_under5"] = np.log(frame["under5"])
    frame["log_gdp"] = np.log(frame["gdp_ppp"])

display(data_2022[MAIN_VARIABLES].describe().round(2))

raw_cols = ["maternal", "under5", "gdp_ppp"]
log_cols = ["log_maternal", "log_under5", "log_gdp"]
skewness = pd.DataFrame({
    "raw": data_2022[raw_cols].skew(),
    "log": data_2022[log_cols].skew().set_axis(raw_cols),
}).round(2)

display(skewness)

Empowerment is moderately negatively correlated with maternal (-0.44) and under-5 mortality (-0.57) and positively with log GDP (0.54), so income is a plausible confounder. Log GDP is much more strongly correlated with both outcomes (-0.85 and -0.88).

In [ ]:
ANALYSIS_VARS = ["v2x_gender", "log_maternal", "log_under5", "log_gdp"]
corr = data_2022[ANALYSIS_VARS].corr(method="spearman").round(2)
display(corr)

Without controls, a 0.1-point higher empowerment score is associated with roughly 24-25% lower mortality in 2022. Once log GDP is included, the maternal coefficient falls to about zero (0.13, 95% CI -0.62 to 0.88), while under-5 mortality remains negatively associated (-0.78, CI -1.28 to -0.28), about 7.5% lower per 0.1 points. A 1% higher GDP per capita is associated with about 1.1% (maternal) and 0.8% (under-5) lower mortality. The 2023 estimates show the same pattern.

In [ ]:
def fit_ols(outcome, data, with_gdp):
    """Fit OLS of a log outcome on empowerment (optionally + log GDP)."""
    formula = f"{outcome} ~ v2x_gender" + (" + log_gdp" if with_gdp else "")
    return smf.ols(formula, data=data).fit(cov_type="HC3")


def summarise(model, outcome, spec, year):
    """Return the empowerment coefficient and fit statistics as a dict."""
    ci = model.conf_int().loc["v2x_gender"]
    b = model.params["v2x_gender"]
    return {
        "year": year, "outcome": outcome, "model": spec,
        "coef_empowerment": b, "ci_low": ci[0], "ci_high": ci[1],
        "p_value": model.pvalues["v2x_gender"],
        "pct_change_per_0.1": (np.exp(0.1 * b) - 1) * 100,
        "r_squared": model.rsquared, "n": int(model.nobs),
    }


rows, models = [], {}
for year, data in [(2022, data_2022), (2023, data_2023)]:
    for outcome in ["log_maternal", "log_under5"]:
        for with_gdp, spec in [(False, "empowerment only"),
                               (True, "+ log GDP")]:
            fitted = fit_ols(outcome, data, with_gdp)
            models[(year, outcome, spec)] = fitted
            rows.append(summarise(fitted, outcome, spec, year))

results = pd.DataFrame(rows)

display(results.round(3))
print(models[(2022, "log_under5", "+ log GDP")].summary().tables[1])

In [ ]:
gdp_effects = pd.DataFrame([
    {"year": y, "outcome": o,
     "gdp_coef": models[(y, o, "+ log GDP")].params["log_gdp"],
     "p_value": models[(y, o, "+ log GDP")].pvalues["log_gdp"]}
    for y in (2022, 2023) for o in ("log_maternal", "log_under5")
])

display(gdp_effects.round(3))

With year fixed effects, the under-5 association persists (-1.14, p < 0.001) and the maternal one is borderline (p = 0.07). With country fixed effects, both estimates are close to zero (p = 0.56 and 0.40), so the association is driven by differences between countries rather than changes within them. The empowerment index moves slowly within countries, so this test has limited power.

In [ ]:
groups = pd.factorize(complete_panel["country_code"])[0]
panel_rows = []
for outcome in ["log_maternal", "log_under5"]:
    for label, extra in [("year FE", " + C(year)"),
                         ("country + year FE",
                          " + C(year) + C(country_code)")]:
        fitted = smf.ols(f"{outcome} ~ v2x_gender + log_gdp{extra}",
                         data=complete_panel).fit(
            cov_type="cluster", cov_kwds={"groups": groups})
        ci = fitted.conf_int().loc["v2x_gender"]
        panel_rows.append({
            "outcome": outcome, "fixed effects": label,
            "coef": fitted.params["v2x_gender"],
            "ci_low": ci[0], "ci_high": ci[1],
            "p_value": fitted.pvalues["v2x_gender"],
            "n": int(fitted.nobs),
        })
display(pd.DataFrame(panel_rows).round(3))

only_2022 = set(data_2022["country_code"]) - set(data_2023["country_code"])
only_2023 = set(data_2023["country_code"]) - set(data_2022["country_code"])
print("Only in 2022:", sorted(only_2022), "| only in 2023:", sorted(only_2023))

## 5. Visualization

**Figure 1: Distributions of the main variables in 2022, raw (top) and log-transformed (bottom).**

Each panel is a histogram of the 167 countries in the 2022 sample. The raw variables are strongly right-skewed: most countries have low mortality and moderate income, while a few have very high values. The data analysis showed that maternal mortality ranges from 1 to 1,016 per 100,000 (mean 124, median 44), under-five mortality from 2.2 to 424.6 per 1,000 (mean 28.4, median 14.7), and GDP per capita from about 1,000 to 135,000 international dollars (mean 25,177, median 15,350). The under-five maximum belongs to the Central African Republic and is far above the next highest value (143 in Somalia); the country's own series fluctuates sharply between years, so we keep it but check its influence in Section 4.3. After the log transformation the long right tails disappear (skewness -0.08, 0.12 and -0.24), but the distributions are broad and uneven rather than bell-shaped, which suggests distinct groups of lower- and higher-income countries. We therefore use log variables to limit the influence of extreme values, and use heteroskedasticity-robust standard errors rather than assuming normally distributed errors.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 6))
panels = [("maternal", "log_maternal", "Maternal mortality"),
          ("under5", "log_under5", "Under-5 mortality"),
          ("gdp_ppp", "log_gdp", "GDP per capita (PPP)")]

for j, (raw, logged, label) in enumerate(panels):
    sns.histplot(data_2022[raw], bins=25, ax=axes[0, j])
    axes[0, j].set_title(f"{label}: raw")
    sns.histplot(data_2022[logged], bins=25, ax=axes[1, j])
    axes[1, j].set_title(f"{label}: log")

plt.tight_layout()
plt.show()

**Figure 2: Women's political empowerment and mortality across countries, 2022.**

Each dot is a country. The red line is an ordinary least squares fit of log mortality on the empowerment index alone, with a shaded 95% confidence band.
Both lines slope downward: countries with higher empowerment tend to have lower maternal mortality (slope about -2.8, R-squared 0.11) and lower under-five
mortality (slope about -2.9, R-squared 0.22). The relationship is clearer for under-five mortality. Most countries have high scores (112 of 167 are above
0.7), while only 12 score below 0.4, so the line and its wide confidence band at the left rest on few observations. Among high-empowerment countries,
mortality varies widely (log maternal mortality from about 1 to over 6), so empowerment alone does not separate them. The point at the top of the right
panel is the Central African Republic. These lines do not account for income, so they should not be read as the effect of empowerment; the regression models
in Section 4 add GDP per capita.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (col, label) in zip(axes, [
        ("log_maternal", "log maternal mortality"),
        ("log_under5", "log under-5 mortality")]):
    sns.regplot(data=data_2022, x="v2x_gender", y=col,
                scatter_kws={"alpha": 0.6, "s": 25},
                line_kws={"color": "crimson"}, ax=ax)
    ax.set_xlabel("Women's political empowerment index (0-1)")
    ax.set_ylabel(label)
    ax.set_title(f"2022: {label}")
    
plt.tight_layout()
plt.show()

**Figure 3: Spearman rank correlations between the main variables, 2022.**

Each cell shows the rank correlation between two variables, from -1 (perfect negative) to +1 (perfect positive). Empowerment is moderately negatively correlated with maternal mortality (-0.44) and under-five mortality (-0.57), and positively with GDP per capita (0.54). GDP is much more stronglycorrelated with b oth outcomes (-0.85 and -0.88), and the two mortality
measures are highly correlated with each other (0.90). Because empowerment and income move together, income is a plausible confounder, and the empowerment association with mortality may partly reflect income. Spearman correlations are used because they do not assume a linear relationship and are less sensitive to outliers.

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(corr, annot=True, cmap="vlag", vmin=-1, vmax=1, ax=ax)
ax.set_title("Correlation heatmap, 2022")
plt.tight_layout()
plt.show()

**Figure 4: Estimated association between empowerment and log mortality, by outcome, year and model.**

Each point is the regression coefficient on the empowerment index (per 1.0-point increase), and the horizontal bars are 95% confidence intervals using robust (HC3) standard errors. The dashed line at zero marks no association. Without controls, all estimates are clearly negative (for example -2.81 for maternal and -2.88 for under-five mortality in 2022). After adding log GDP, the maternal mortality estimates move to about zero and their intervals include zero (0.13 in 2022, -0.11 in 2023). The under-five estimates shrink but remain negative, with intervals that exclude zero (-0.78 in 2022, -0.89 in 2023). The 2022 and 2023 results are very similar, which supports the robustness of the findings to the choice of year.

In [ ]:
plot_df = results.assign(
    label=lambda d: d["year"].astype(str) + " | "
    + d["outcome"].str.replace("log_", "") + " | " + d["model"])
y_pos = np.arange(len(plot_df))

fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(
    plot_df["coef_empowerment"], y_pos,
    xerr=[plot_df["coef_empowerment"] - plot_df["ci_low"],
          plot_df["ci_high"] - plot_df["coef_empowerment"]],
    fmt="o", capsize=3)
ax.axvline(0, color="grey", linestyle="--")
ax.set_yticks(y_pos)
ax.set_yticklabels(plot_df["label"])
ax.invert_yaxis()
ax.set_xlabel("Empowerment coefficient (log points per 1.0 index, 95% CI)")
plt.tight_layout()
plt.show()

**Figure 5: Median mortality over time by empowerment group, 2000-2023.**

Countries are divided into three equally sized groups (about 55 each) by their 2022 empowerment score, and the lines show the median mortality of each group per year. The groups are fixed, so a country does not change group over time. Mortality fell in every group. Median under-five mortality dropped from about 70 to 29 per 1,000 in the low group (-59%), from 71 to 26 in the middle group (-63%) and from 8 to 4 in the high group (-47%). Median maternal mortality fell from 166 to 80 (low), 206 to 99 (middle) and 13 to 8 (high) per 100,000.
The high-empowerment group started at much lower levels, so the gap in absolute terms narrowed over time, while the low and middle groups remain far above it. Note that a few countries lack data in early years, so the group composition changes slightly (for example 52 low-group countries have data in 2000 versus 55 in 2023).

In [ ]:
# sort the 167 countries by empowerment score and cut them into three equal-sized groups of about 55 to 56 countries:
tercile = pd.qcut(data_2022.set_index("country_code")["v2x_gender"], 3, labels=["Lower group", "Middle group", "Higher group"])

trend = (complete_panel
         .assign(group=complete_panel["country_code"].map(tercile))
         .dropna(subset=["group"])
         .groupby(["year", "group"], observed=True)[["under5", "maternal"]]
         .median().reset_index())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, col, label in zip(axes, ["under5", "maternal"],
                          ["Under-5 mortality (per 1,000)",
                           "Maternal mortality (per 100,000)"]):
    sns.lineplot(data=trend, x="year", y=col, hue="group", ax=ax)
    ax.legend(title="Empowerment group (2022)")
    ax.set_ylabel("Median " + label)
    ax.set_title(label)

plt.tight_layout()
plt.show()

**Figure 6: Under-five mortality by empowerment quartile, 2022.**

Countries are grouped into quartiles of the Women's Political Empowerment Index (Q1 = lowest) and shown in the box plots below. 
Boxes show the interquartile range, the line is the median, the whiskers extend to 1.5 times the interquartile range, and dots mark outliers.
The vertical axis is logarithmic. Median under-five mortality falls from about 29 per 1,000 in the lowest quartile to about 4 in the highest. 
This comparison does not control for income, so it should be read together with the regression results in Section 4. 2022 is the primary sample and 2023 is checked in the
sensitivity analysis.

In [ ]:
quartiles = data_2022.assign(quartile=pd.qcut(
    data_2022["v2x_gender"], 4,
    labels=["Q1 (lowest)", "Q2", "Q3", "Q4 (highest)"]))
fig, ax = plt.subplots(figsize=(6, 4.5))
sns.boxplot(data=quartiles, x="quartile", y="under5", ax=ax)
ax.set_yscale("log")
ax.set_xlabel("Empowerment quartile (2022)")
ax.set_ylabel("Under-5 mortality (per 1,000, log scale)")
plt.tight_layout()
plt.show()

## 6. Discussion

## 7. Conclusion

## 8. Reflection

## 9. AI Use Statement